# Search for optimal partition with minimal description length based on Map equation
The original paper proposes a greedy search followed by simulated annealing, but in the 2009 publication "The map equation" by M. Rosvall1, D. Axelsson, and C.T. Bergstrom, a method supposedly as fast but more accurate than the method from the OG paper was introduced. It's described as follows:

## 5 Fast stochastic and recursive search algorithm
Any greedy (fast but inaccurate) or Monte Carlo-based (accurate but slow) approach can be 
used to minimize the map equation. To provide a good balance between the two extremes, 
we have developed a fast stochastic and recursive search algorithm, implemented it in C++, 
and made it available online both for directed and undirected weighted networks [25]. As a 
reference, the new algorithm is as fast as the previous high-speed algorithms (the greedy search 
presented in the supporting appendix of Ref. [14]), which were based on the method introduced 
in Ref. [26] and reﬁned in Ref. [27]. At the same time, it is also more accurate than our previous 
high-accuracy algorithm (a simulated annealing approach) presented in the same supporting 
appendix.

The core of the algorithm follows closely the method presented in Ref. [28]: neighboring 
nodes are joined into modules, which subsequently are joined into supermodules and so on. 
First, each node is assigned to its own module. Then, in random sequential order, each node is 
moved to the neighboring module that results in the largest decrease of the map equation. If 
no move results in a decrease of the map equation, the node stays in its original module. This 
procedure is repeated, each time in a new random sequential order, until no move generates a 
decrease of the map equation. Now the network is rebuilt, with the modules of the last level 
forming the nodes at this level. And exactly as at the previous level, the nodes are joined into 
modules. This hierarchical rebuilding of the network is repeated until the map equation cannot 
be reduced further. Except for the random sequence order, this is the algorithm described in 
Ref. [28].

With this algorithm, a fairly good clustering of the network can be found in a very short 
time. Let us call this the core algorithm and see how it can be improved. The nodes assigned to 
the same module are forced to move jointly when the network is rebuilt. As a result, what was
an optimal move early in the algorithm might have the opposite eﬀect later in the algorithm. 
Because two or more modules that merge together and form one single module when the network 
is rebuilt can never be separated again in this algorithm, the accuracy can be improved by 
breaking the modules of the ﬁnal state of the core algorithm in either of the two following ways:

*Submodule movements*. First, each cluster is treated as a network on its own and the main 
algorithm is applied to this network. This procedure generates one or more submodules 
for each module. Then all submodules are moved back to their respective modules of the 
previous step. At this stage, with the same partition as in the previous step but with 
each submodule being freely movable between the modules, the main algorithm is re-applied.

*Single-node movements*. First, each node is re-assigned to be the sole member of its own 
module, in order to allow for single-node movements. Then all nodes are moved back to 
their respective modules of the previous step. At this stage, with the same partition as in 
the previous step but with each single node being freely movable between the modules, the 
main algorithm is re-applied.
In practice, we repeat the two extensions to the core algorithm in sequence and as long as the 
clustering is improved. Moreover, we apply the submodule movements recursively. That is, to 
ﬁnd the submodules to be moved, the algorithm ﬁrst splits the submodules into subsubmodules, 
subsubsubmodules, and so on until no further splits are possible. Finally, because the algorithm 
is stochastic and fast, we can restart the algorithm from scratch every time the clustering cannot 
be improved further and the algorithm stops. The implementation is straightforward and, by 
repeating the search more than once, 100 times or more if possible, the ﬁnal partition is less 
likely to correspond to a local minimum. For each iteration, we record the clustering if the 
description length is shorter than the previously shortest description length. In practice, for 
networks with on the order of 10,000 nodes and 1,000,000 directed and weighted links, each 
iteration takes about 5 seconds on a modern PC

Phase 1:
- initially assign each node to a different communitiy ($N$ nodes -> $N$ communities)
- for each node i, consider neighbours j, check if/how much removing i from its community and putting it into j's community improves (decreases) description length
- choose neighbour merge with best improvement. If no improvement possible, i stays in its community.
- repeat sequentially for all nodes until no further improvement can be made -> Phase 1 done!

Phase 2:
- build a new network whose nodes are the communities found in phase 1
- weights of links between new nodes correspond to sum of weight of links between old nodes in the corresponding 2 communities; links between nodes of the same community lead to self-loops for this community in the new network -> Phase 2 done! Can reapply Phase 1 to new network.

Iterate Phase 1 + Phase 2 until no further improvements are obtained.
Refinement possible with submodule/single-module movements -> can be performed in sequence until no more improvements.

The following functions are (presumably) needed to tackle this:
- `update_node_move_description_length()`: Computes the change in description length for a given community partition in the event that a single node is moved from its community to another community. (for phase 1) -> DONE
- `node_movement_optimization()`: Basically runs the Phase 1. Sequentially goes through nodes in random order, finds the optimal move using `update_node_move_description_length()`, and repeats until no improving moves can be made.
- `compress_network()`: Implements Phase 2. Takes the community assignment and transforms it into a new network with nodes corresponding to the communities, and aggregates the weights of the within- and between-community links into self-loops and links between nodes on the new network.

In [2]:
# library imports
import igraph as ig
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import timeit
import warnings
import random
import infomap_funcs as inf

Let's try to do the node movement optimization:
(Some mild confusion: One of the references makes it sound as though we keep sequentially going through all nodes again until no more improvements are possible, the other one suggests that we keep track of candidate nodes and, if at any point, a node has no further moves for improvement it is removed from the candidate pool, which unlike the first option sounds like it may neglegt combinations where after a change in constellation a node that previously didn't have an improving move might now have one with the new, changed modules. We'll see? I'll start implementing option one, and then we can see if the candidate pool makes sense or not.)

In [19]:
g_test = inf.generate_sbm(n=100, c=4, p_in=0.25, p_out=0.01, directed=False, weighted=True)
nb=g_test.neighborhood(mindist=1)

In [20]:
node=5
print(g_test.neighbors(node))
print(nb[node])


[9, 10, 19, 20, 23]
[9, 10, 19, 20, 23]


In [ ]:
def node_movement_optimization(g):
    nodes = g.vs.indices # get list of nodes
    neighborhood = g.neighborhood(mindist=1) # get list of neighbours for all nodes

    # initialize community partition with each node being its own community
    communities = range(g.vcount()) # start with each node assigned to its own community

    # compute description length including some intermediate terms:
    L, p, p_mod, exit_data = inf.compute_description_length(g, communities, returnTerms=True)

    optimizable=True
    while optimizable: # while there are still improvements via node moves:
        # randomize node sequence
        random.shuffle(nodes)

        # for each node go through neighbours (if different community(?))
        for n in nodes:
            neighbors = neighborhood[n] # get neighbors of node
            nb_comms = communities[neighbors] # get communties of neighbors
            L_best, communities_best, exits_data_best = L, communities, exit_data
            # go through unique neighbouring communities  
            for nbc in np.unique(nb_comms):
                # get new description length for assigning node to different community
                L_new, communities_new, exit_data_new = inf.update_node_move_description_length(g, p, p_mod, exit_data, node, nbc, returnTerms=True)
                if L_new is not None and L_new < L_best: # if better description length
                    # update with best constellation
                    L_best, communities_best, exits_data_best = L_new, communities_new, exit_data_new

            # move to community with best improvement if possible, otherwise leave where it is
            
    # repeat until no more optimizing moves

    communities_optimized = None # placeholder
    return communities_optimized


NameError: name 'NaN' is not defined